In [1]:
import numpy as np
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

import torchvision
import torchvision.transforms as transforms

In [2]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

In [4]:
train_data = torchvision.datasets.CIFAR10(root='./data', train=True, transform=transform, download=True)
test_data = torchvision.datasets.CIFAR10(root='./data', train=False, transform=transform, download=True)



In [5]:
train_loader = torch.utils.data.DataLoader(train_data, batch_size=32, shuffle=True, num_workers=2)
test_loader = torch.utils.data.DataLoader(test_data, batch_size=32, shuffle=True, num_workers=2)

In [6]:
image, label = train_data[0]

In [7]:
image.size()

torch.Size([3, 32, 32])

In [8]:
class_names = ['plane', 'car', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']

In [9]:
class NeuralNet(nn.Module):

    def __init__(self):
        super().__init__()

        self.conv1 = nn.Conv2d(3, 12, 5) # input size - kernel size 32 - 5 = 27/1(stride) = 27 + 1 = 28, after convolution (12,28,28)
        self.pool = nn.MaxPool2d(2, 2) # (12, 14, 14)
        self.conv2 = nn.Conv2d(12, 24, 5) # (24, 10, 10) -> (24, 5, 5) -> Flatten (24 * 5 * 5)
        self.fc1 = nn.Linear(24 * 5 * 5, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = torch.flatten(x, 1)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)

        return x

In [10]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

net = NeuralNet().to(device)
loss_function = nn.CrossEntropyLoss()
optimizer = optim.SGD(net.parameters(), lr=0.001, momentum=0.9)

In [11]:
for epoch in range(30):
    print(f'Training epoch {epoch}..')
    running_loss = 0.0

    for i, data in enumerate(train_loader):
        inputs, labels = data
        inputs = inputs.to(device)      # ← critical
        labels = labels.to(device)      # ← critical

        optimizer.zero_grad()

        outputs = net(inputs)
        loss = loss_function(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(f'Loss: {running_loss / len(train_loader):.4f}')

Training epoch 0..
Loss: 2.2824
Training epoch 1..
Loss: 1.8622
Training epoch 2..
Loss: 1.5612
Training epoch 3..
Loss: 1.4253
Training epoch 4..
Loss: 1.3244
Training epoch 5..
Loss: 1.2462
Training epoch 6..
Loss: 1.1700
Training epoch 7..
Loss: 1.1002
Training epoch 8..
Loss: 1.0487
Training epoch 9..
Loss: 0.9973
Training epoch 10..
Loss: 0.9574
Training epoch 11..
Loss: 0.9182
Training epoch 12..
Loss: 0.8815
Training epoch 13..
Loss: 0.8474
Training epoch 14..
Loss: 0.8141
Training epoch 15..
Loss: 0.7844
Training epoch 16..
Loss: 0.7575
Training epoch 17..
Loss: 0.7248
Training epoch 18..
Loss: 0.6999
Training epoch 19..
Loss: 0.6767
Training epoch 20..
Loss: 0.6536
Training epoch 21..
Loss: 0.6259
Training epoch 22..
Loss: 0.6071
Training epoch 23..
Loss: 0.5834
Training epoch 24..
Loss: 0.5604
Training epoch 25..
Loss: 0.5410
Training epoch 26..
Loss: 0.5169
Training epoch 27..
Loss: 0.4993
Training epoch 28..
Loss: 0.4793
Training epoch 29..
Loss: 0.4583


In [12]:
torch.save(net.state_dict(), 'trained_net.pth')

In [20]:
trained_model = NeuralNet()
trained_model = trained_model.to(device)
trained_model.load_state_dict(torch.load('trained_net.pth'))

<All keys matched successfully>

In [21]:
correct = 0
total = 0
trained_model.eval()

with torch.no_grad():
    for data in test_loader:
        images, labels = data
        images = images.to(device)
        labels = labels.to(device)

        outputs = trained_model(images)
        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total
print(f'Accuracy: {accuracy:.2f}%')

Accuracy: 68.32%
